# 04 – Machine Learning & Clustering

Apply K-means, DBSCAN, and Random Forest classification to soundscape
feature vectors.  Explore the acoustic feature space with PCA.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

from src.ml_clustering.clustering import (
    kmeans_clustering, dbscan_clustering, silhouette_analysis
)
from src.ml_clustering.classification import (
    train_soundscape_classifier, detect_anomalies, feature_importance_analysis
)
from src.visualization.plots import plot_pca, plot_confusion_matrix
from src.utils.logger import setup_logger

setup_logger(level='INFO')
print('Imports OK')

## 1. Synthetic Feature Matrix

In [ ]:
rng = np.random.default_rng(42)
n = 120

# Four synthetic soundscape types with distinct signatures
labels_true = np.repeat(['urban', 'forest', 'riverbank', 'agricultural'], n // 4)

X = np.vstack([
    rng.multivariate_normal([100, -0.2, 0.8, 50,  -20], np.eye(5) * 25, n // 4),  # urban
    rng.multivariate_normal([600,  0.6, 2.5, 200, -35], np.eye(5) * 25, n // 4),  # forest
    rng.multivariate_normal([400,  0.3, 1.8, 120, -30], np.eye(5) * 25, n // 4),  # riverbank
    rng.multivariate_normal([200, -0.1, 1.2, 80,  -25], np.eye(5) * 25, n // 4),  # agricultural
])

feature_names = ['aci', 'ndsi', 'adi', 'bi', 'rms_db']
df = pd.DataFrame(X, columns=feature_names)
df['label'] = labels_true
print(df.head())

## 2. K-means Clustering

In [ ]:
km_labels, km_model = kmeans_clustering(X, n_clusters=4)
print('Cluster distribution:', dict(zip(*np.unique(km_labels, return_counts=True))))

## 3. Silhouette Analysis

In [ ]:
sil_df = silhouette_analysis(X, k_range=range(2, 8))
print(sil_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sil_df['k'], sil_df['silhouette'], marker='o')
ax.set_xlabel('Number of clusters k')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette Analysis')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. DBSCAN Clustering

In [ ]:
db_labels, _ = dbscan_clustering(X, eps=1.5, min_samples=5)
unique, counts = np.unique(db_labels, return_counts=True)
print('DBSCAN clusters:', dict(zip(unique, counts)))

## 5. PCA Visualisation

In [ ]:
fig = plot_pca(X, labels=km_labels, feature_names=feature_names,
               title='PCA – K-means Cluster Labels')
plt.show()

## 6. Random Forest Classification

In [ ]:
clf_result = train_soundscape_classifier(X, labels_true, cv_folds=3)
print(f"Accuracy : {clf_result['accuracy']:.4f}")
print(f"CV scores: {clf_result['cv_scores'].mean():.4f} ± {clf_result['cv_scores'].std():.4f}")

In [ ]:
fig = plot_confusion_matrix(
    clf_result['confusion_matrix'],
    class_names=list(clf_result['encoder'].classes_),
    title='Confusion Matrix – Soundscape Classification'
)
plt.show()

## 7. Feature Importance

In [ ]:
fi_df = feature_importance_analysis(
    clf_result['model'], feature_names=feature_names
)
print(fi_df)

## 8. Anomaly Detection

In [ ]:
anomaly_labels = detect_anomalies(X, contamination=0.05)
n_anomalies = int(np.sum(anomaly_labels == -1))
print(f'Anomalies detected: {n_anomalies} / {len(anomaly_labels)}')